# 11 - AgentCore Capabilities: Code Execution, Browser & Workload Identity

Demonstrates the three AgentCore capability resources deployed in `OrgIQ-AgentCoreCapabilities`:

| Capability | CDK Resource | Use case |
|---|---|---|
| **Code Interpreter** | `CfnCodeInterpreterCustom` | Sandboxed Python for NER, PageRank |
| **Browser** | `CfnBrowserCustom` + `CfnBrowserProfile` | Web research, profile enrichment |
| **Workload Identity** | `CfnWorkloadIdentity` | OAuth2 token federation for portal |

Prerequisites: `OrgIQ-AgentCoreCapabilities` deployed — `CODE_INTERPRETER_ID`, `BROWSER_ID`, and `WORKLOAD_IDENTITY_ARN` in `config.json`.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import uuid
from pathlib import Path

config = json.loads(Path('config.json').read_text())
REGION = config.get('REGION', 'us-east-1')
PREFIX = config.get('PROJECT_PREFIX', 'orgiq')
MODEL_ID = config.get('MODEL_ID', 'amazon.nova-pro-v1:0')
CODE_INTERPRETER_ID = config.get('CODE_INTERPRETER_ID', '')
BROWSER_ID          = config.get('BROWSER_ID', '')
WORKLOAD_IDENTITY_ARN = config.get('WORKLOAD_IDENTITY_ARN', '')

agentcore = boto3.client('bedrock-agentcore', region_name=REGION)
agentcore_ctrl = boto3.client('bedrock-agentcore-control', region_name=REGION)

print(f'Code Interpreter: {CODE_INTERPRETER_ID or "not configured"}')
print(f'Browser:          {BROWSER_ID or "not configured"}')
print(f'Workload Identity: {WORKLOAD_IDENTITY_ARN or "not configured"}')


## What Is AgentCore Code Execution?

Amazon Bedrock AgentCore provides **sandboxed Python execution environments** that agents can use to run arbitrary code safely during a conversation. Each invocation spins up an isolated container with:

- **Python 3.12** with a curated set of data-science packages (`pandas`, `numpy`, `matplotlib`, `scipy`, `statsmodels`, `scikit-learn`)
- **No outbound network access** — the sandbox cannot make HTTP calls, reach AWS APIs, or contact external services
- **Ephemeral filesystem** — files written during execution can be retrieved as base64-encoded artifacts; nothing persists between sessions
- **Hard time limit** — executions time out after ~30 s; long-running loops are killed
- **No subprocess / shell** — `subprocess`, `os.system`, and `eval` on untrusted input are blocked

### Why use it in OrgIQ?

| Use case | How code execution helps |
|---|---|
| Expertise score statistics | Compute mean / median / stdev on live DynamoDB data |
| Distribution histograms | Generate ASCII or matplotlib charts in-agent |
| Ad-hoc data wrangling | Filter, group, sort org data without hardcoding logic |
| CSV / JSON exports | Write structured artifacts the caller can download |
| Reproducible calculations | Audit trail: the code that produced a number is logged |

## Step 2: Direct API Invocation

The `bedrock-agentcore` boto3 client exposes a Code Execution API.  
The exact shape of the response depends on the SDK version installed, so we wrap the call in a helper and print the raw result for inspection.

In [ ]:
# NOTE: The Code Execution API surface is versioned.  If the call below raises
# a NoSuchOperation or similar error, check `bedrock_agentcore.__version__` and
# the AWS documentation for the exact operation name in your SDK version.

try:
    import bedrock_agentcore  # type: ignore
    print(f"bedrock-agentcore SDK version: {bedrock_agentcore.__version__}")
except ImportError:
    print("\u26a0\ufe0f  bedrock-agentcore not installed; pip install bedrock-agentcore>=1.9")

# Build the low-level client
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

# Sample Python snippet to execute in the sandbox
SAMPLE_CODE = """
import math, json

# Simple calculation: expertise score normalization demo
raw_scores = [0.45, 0.72, 0.88, 0.91, 0.60, 0.55, 0.78, 0.83, 0.50, 0.95]
min_s, max_s = min(raw_scores), max(raw_scores)
normalized = [(s - min_s) / (max_s - min_s) for s in raw_scores]

result = {
    "raw_scores": raw_scores,
    "min": min_s,
    "max": max_s,
    "mean": sum(raw_scores) / len(raw_scores),
    "normalized": [round(n, 4) for n in normalized],
}
print(json.dumps(result, indent=2))
"""

session_id = str(uuid.uuid4())

try:
    # Attempt invocation — operation name may vary across SDK versions
    # Possibilities: invoke_code_interpreter / create_code_interpreter_session + invoke_code_interpreter
    response = agentcore_client.invoke_code_interpreter(
        sessionId=session_id,
        code=SAMPLE_CODE,
        language="PYTHON",
    )

    output = response.get("output", {})
    print("--- stdout ---")
    print(output.get("stdout", ""))
    if output.get("stderr"):
        print("--- stderr ---")
        print(output["stderr"])
    print(f"\n\u2713 Direct API invocation succeeded (session={session_id[:8]}...)")

except agentcore_client.exceptions.ClientError as exc:
    error_code = exc.response["Error"]["Code"]
    print(f"\u26a0\ufe0f  ClientError ({error_code}): {exc}")
    print("This is expected if Code Execution is not yet enabled in your region")
    print("or if the API surface differs in your SDK version.")
except Exception as exc:
    print(f"\u26a0\ufe0f  {type(exc).__name__}: {exc}")
    print("Continuing — Steps 3+ use the Strands Agent approach which is more portable.")

## Step 3: Strands Agent with `code_interpreter` Tool

The Strands Agents framework ships a built-in `code_interpreter` tool that wraps the AgentCore Code Execution API.  
Passing it to `Agent(tools=[...])` gives the model the ability to write and run Python code autonomously.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

try:
    from strands.tools import code_interpreter  # built-in tool  # type: ignore
    CODE_INTERPRETER_AVAILABLE = True
    print("\u2713 code_interpreter tool imported from strands.tools")
except ImportError:
    CODE_INTERPRETER_AVAILABLE = False
    print("\u26a0\ufe0f  code_interpreter not found in this strands-agents version")
    print("    Falling back to a custom @tool wrapper around the boto3 client.")

model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION
)

CODE_SYSTEM_PROMPT = """You are OrgIQ Analyst, an organizational intelligence assistant with
the ability to execute Python code in a secure sandbox.

When asked to perform calculations, generate statistics, or analyse data:
1. Write clear, readable Python code
2. Print results so they appear in the output
3. Add brief comments explaining what each block does
4. Return a plain-language summary after the code block

The sandbox has: pandas, numpy, scipy, statistics, json, math, csv, io.
It does NOT have: network access, boto3, requests, or subprocess.
"""

if CODE_INTERPRETER_AVAILABLE:
    analyst_agent = Agent(
        model=model,
        system_prompt=CODE_SYSTEM_PROMPT,
        tools=[code_interpreter],
    )
    print("\u2713 Analyst agent created with code_interpreter tool")
else:
    # Fallback: define a lightweight @tool that delegates to the boto3 client
    from strands import tool

    @tool
    def run_python(code: str) -> str:
        """Execute a Python snippet in an AgentCore sandbox and return stdout.

        Args:
            code: Valid Python 3.12 source code to run in the sandbox.
        """
        try:
            sid = str(uuid.uuid4())
            resp = agentcore_client.invoke_code_interpreter(
                sessionId=sid, code=code, language="PYTHON"
            )
            out = resp.get("output", {})
            result = out.get("stdout", "")
            if out.get("stderr"):
                result += f"\n[stderr]\n{out['stderr']}"
            return result or "(no output)"
        except Exception as exc:
            return f"ERROR: {exc}"

    analyst_agent = Agent(
        model=model,
        system_prompt=CODE_SYSTEM_PROMPT,
        tools=[run_python],
    )
    print("\u2713 Analyst agent created with fallback run_python tool")

In [ ]:
# Quick smoke-test: ask the agent to do a simple calculation
SAMPLE_ORG_JSON = json.dumps({
    "organization": PREFIX,
    "people": [
        {"name": "Alice Chen",    "title": "Staff Engineer",   "expertise_score": 0.91, "topics": 5, "documents": 4},
        {"name": "Bob Kumar",     "title": "ML Engineer",      "expertise_score": 0.95, "topics": 6, "documents": 3},
        {"name": "Carol Santos",  "title": "SRE",              "expertise_score": 0.78, "topics": 4, "documents": 2},
        {"name": "Dave Park",     "title": "Platform Engineer","expertise_score": 0.83, "topics": 5, "documents": 5},
        {"name": "Eve Nakamura",  "title": "Security Eng",     "expertise_score": 0.88, "topics": 4, "documents": 3},
        {"name": "Frank Osei",   "title": "Data Scientist",   "expertise_score": 0.72, "topics": 3, "documents": 2},
        {"name": "Grace Li",     "title": "Backend Engineer", "expertise_score": 0.60, "topics": 3, "documents": 1},
        {"name": "Hiro Tanaka",  "title": "DevOps Engineer",  "expertise_score": 0.55, "topics": 2, "documents": 1},
        {"name": "Iris Johansson","title": "Frontend Engineer","expertise_score": 0.50, "topics": 2, "documents": 0},
        {"name": "James Okafor", "title": "Junior Engineer",  "expertise_score": 0.45, "topics": 1, "documents": 0},
    ]
}, indent=2)

smoke_prompt = (
    f"Here is a snapshot of our org data:\n```json\n{SAMPLE_ORG_JSON}\n```\n\n"
    "Write Python code to calculate the average expertise_score and print "
    "the top 3 people by score. Keep the code short."
)

print("Sending smoke-test query...\n")
response = analyst_agent(smoke_prompt)
print("\n--- Agent response ---")
print(response)
print("\n\u2713 Smoke test complete")

## Step 4: Analyze Scoring Data from DynamoDB

Read the live expertise scores from the `orgiq-scores` DynamoDB table, then pass them to the code-interpreter agent for statistical analysis.

In [ ]:
import decimal

SCORES_TABLE = f"{PREFIX}-scores"
dynamodb = boto3.resource("dynamodb", region_name=REGION)

def load_scores_from_dynamo(table_name: str) -> list[dict]:
    """Scan the scores table and return a list of person score records."""
    table = dynamodb.Table(table_name)
    items = []
    paginator = table.meta.client.get_paginator("scan")
    for page in paginator.paginate(TableName=table_name):
        items.extend(page["Items"])
    # Decimal -> float for JSON serialization
    def _dec(obj):
        if isinstance(obj, list):   return [_dec(i) for i in obj]
        if isinstance(obj, dict):   return {k: _dec(v) for k, v in obj.items()}
        if isinstance(obj, decimal.Decimal): return float(obj)
        return obj
    return _dec(items)

try:
    scores_data = load_scores_from_dynamo(SCORES_TABLE)
    print(f"\u2713 Loaded {len(scores_data)} records from {SCORES_TABLE}")
    if scores_data:
        print(f"  Sample record keys: {list(scores_data[0].keys())}")
except Exception as exc:
    print(f"\u26a0\ufe0f  Could not read {SCORES_TABLE}: {exc}")
    print("  Using embedded sample data instead.")
    scores_data = json.loads(SAMPLE_ORG_JSON)["people"]  # fallback to smoke-test data

# Normalise: ensure every record has an 'expertise_score' key
# DynamoDB records may use 'composite_score' or 'score' depending on notebook 13
for rec in scores_data:
    if "expertise_score" not in rec:
        rec["expertise_score"] = rec.get("composite_score", rec.get("score", 0.0))

scores_json_str = json.dumps(scores_data, indent=2)
print(f"  Data payload: {len(scores_json_str)} characters")

In [ ]:
# Ask the agent to perform statistical analysis on the real (or sample) scores data
stats_prompt = f"""
Here are the expertise scores for OrgIQ members (JSON):

```json
{scores_json_str}
```

Please write Python code to:
1. Extract the 'name' and 'expertise_score' fields for each person
2. Calculate: mean, median, standard deviation, min, max of expertise_score
3. Print the top 3 people (highest scores) and bottom 3 people (lowest scores)
4. Generate a text histogram of the score distribution using 5 buckets
   (e.g. 0.0-0.2, 0.2-0.4, 0.4-0.6, 0.6-0.8, 0.8-1.0)

Use only the Python standard library (statistics, math) — no pandas.
Print all results clearly labelled.
"""

print("Sending scoring analysis query...\n")
stats_response = analyst_agent(stats_prompt)
print("\n--- Agent response ---")
print(stats_response)
print("\n\u2713 Statistical analysis complete")

## Step 5: File Artifacts from Code Execution

The code execution sandbox supports writing files (CSV, JSON, PNG) which are returned as base64-encoded artifacts in the API response.  
Here we ask the agent to write a CSV export of the scores and then decode it.

In [ ]:
import base64, io, csv

artifact_prompt = f"""
Using this scores data:
```json
{scores_json_str}
```

Write Python code that:
1. Reads the JSON (use json.loads on the string literal)
2. Builds a CSV with columns: name, expertise_score, rank
   (rank = 1 for highest score)
3. Writes the CSV to a file called 'scores_export.csv'
4. Also prints the CSV content to stdout so we can see it

Use only csv, json from the standard library.
"""

print("Requesting CSV artifact generation...\n")
artifact_response = analyst_agent(artifact_prompt)
print("\n--- Agent response ---")
print(artifact_response)

# If the underlying API returned file artifacts, they would be in the raw tool result.
# The Strands code_interpreter tool surfaces them via event.data["artifacts"].
# Example retrieval (works when using the direct boto3 API):
print("\n--- How to retrieve file artifacts via boto3 ---")
print("""
# After a direct invoke_code_interpreter call:
artifacts = response.get("artifacts", [])
for artifact in artifacts:
    filename = artifact["name"]
    content  = base64.b64decode(artifact["content"])
    Path(filename).write_bytes(content)
    print(f"Saved {filename} ({len(content)} bytes)")
""")
print("\u2713 Artifact retrieval pattern shown")

## Step 6: Advanced Analysis — Score Percentiles and Histogram

Demonstrate a more complex analysis request: percentile bands and a visual ASCII chart.

In [ ]:
advanced_prompt = f"""
Analyse this OrgIQ expertise data:
```json
{scores_json_str}
```

Write Python code that produces:

1. **Percentile table** — for each person, compute their percentile rank
   within the group (0=lowest, 100=highest). Print as a table.

2. **ASCII bar chart** — horizontal bars proportional to score, e.g.:
   ```
   Bob Kumar     |################  | 0.95
   Alice Chen    |###############   | 0.91
   ...
   ```
   Use 20 characters as the bar width. Sort by score descending.

3. **Cohort split** — print two lists:
   - High performers: expertise_score >= 0.8
   - Needs development: expertise_score < 0.65

Use only the Python standard library.
"""

print("Requesting advanced analysis...\n")
advanced_response = analyst_agent(advanced_prompt)
print("\n--- Agent response ---")
print(advanced_response)
print("\n\u2713 Advanced analysis complete")

## Safety Boundaries of Code Execution

### What code execution CAN do

| Capability | Notes |
|---|---|
| Pure Python computation | Arithmetic, string ops, data transformations |
| Standard library | `math`, `statistics`, `csv`, `json`, `io`, `pathlib`, `datetime`, `itertools`, `collections` |
| Scientific packages | `numpy`, `pandas`, `scipy`, `statsmodels`, `scikit-learn` |
| Plotting | `matplotlib` — figures are returned as PNG artifacts |
| File writes | Write to the sandbox filesystem; files come back as base64 artifacts |
| stdin / stdout | `print()` output is captured and returned as `stdout` |

### What code execution CANNOT do

| Restriction | Reason |
|---|---|
| No outbound network | `requests`, `urllib`, `socket` are blocked; prevents data exfiltration |
| No AWS API calls | `boto3` is not available in the sandbox; agents cannot escalate permissions |
| No subprocess / shell | `subprocess`, `os.system`, `os.popen` are blocked; no shell injection |
| No persistent filesystem | Sandbox is destroyed after each invocation; files do not survive across calls |
| No GPU | CPU-only execution; very large matrix ops will be slow |
| Hard time limit | ~30 s timeout; infinite loops are killed with a `TimeoutError` |
| No `eval` on untrusted input | Not blocked at the OS level, but the agent's system prompt should prohibit it |

### Defense in depth

Even though the sandbox is isolated, follow these practices in your agents:
1. **Validate agent-generated code** with a guardrail before execution if it handles PII
2. **Pass data as JSON strings**, not raw Python — avoids code injection via data values
3. **Set a system-prompt prohibition** on `eval`, `exec`, and `__import__` on user-supplied strings
4. **Treat stdout as untrusted** — sanitize before rendering in the portal UI

In [ ]:
# Demonstrate that the agent respects safety boundaries
safety_prompt = """
Write Python code that tries to:
1. Make an HTTP request to https://example.com
2. Run a shell command with subprocess

Then explain what happens and why these are blocked.
"""

print("Testing safety boundary awareness...\n")
safety_response = analyst_agent(safety_prompt)
print("\n--- Agent response ---")
print(safety_response)
print("\n\u2713 Safety boundary test complete")

## Step 7: Save to Configuration

## Step 8: Browser Sessions

The `CfnBrowserCustom` resource provisions a managed headless browser that agents can drive.
Sessions persist a `BrowserProfile` so cookies and session state survive across invocations.

**Agents with browser access:** supervisor, search, advisor (as configured in `OrgIQ-AgentCoreCapabilities`).

The browser API follows a start/navigate/interact/stop pattern.
Sessions incur cost while open — always stop them when done.

In [ ]:
# Start a browser session
# NOTE: Exact API method names depend on bedrock-agentcore SDK version
if BROWSER_ID:
    try:
        session = agentcore.start_browser_session(
            browserId=BROWSER_ID,
            profileId=config.get('BROWSER_PROFILE_ID', ''),
            sessionId=str(uuid.uuid4()),
        )
        browser_session_id = session['sessionId']
        print(f'Browser session started: {browser_session_id}')

        # Navigate to a page
        agentcore.navigate_browser_to_page(
            browserId=BROWSER_ID,
            sessionId=browser_session_id,
            url='https://example.com',
        )
        print('Navigated to example.com')

        # Get page source
        result = agentcore.get_browser_page_source(
            browserId=BROWSER_ID,
            sessionId=browser_session_id,
        )
        print(f'Page source length: {len(result.get("pageSource", ""))} bytes')

        # Stop the session
        agentcore.stop_browser_session(
            browserId=BROWSER_ID,
            sessionId=browser_session_id,
        )
        print('Session stopped')
    except Exception as e:
        print(f'Browser API: {e}')
        print('Note: verify bedrock-agentcore SDK version supports browser APIs')
else:
    print('BROWSER_ID not set — deploy OrgIQ-AgentCoreCapabilities first')


## Step 9: Workload Identity (OAuth2 Token Federation)

The `CfnWorkloadIdentity` resource lets the OrgIQ portal and middleware exchange their Cognito/Okta OAuth2 tokens for short-lived AgentCore-scoped credentials — no long-lived access keys embedded in the frontend.

**Flow:**
1. User authenticates with your identity provider (Cognito, Okta, etc.)
2. IdP issues an OAuth2 `access_token`
3. Portal/middleware calls `GetWorkloadAccessTokenForJwt` with that token
4. AgentCore validates the JWT and returns a short-lived workload token
5. That workload token is used to call `InvokeAgentRuntime`

In [ ]:
# Inspect the Workload Identity resource
if WORKLOAD_IDENTITY_ARN:
    try:
        # List workload identities to confirm deployment
        resp = agentcore_ctrl.list_workload_identities()
        identities = resp.get('workloadIdentities', [])
        wi = next((i for i in identities if i['arn'] == WORKLOAD_IDENTITY_ARN), None)
        if wi:
            print(f'Workload Identity: {wi["name"]}')
            print(f'ARN:   {wi["arn"]}')
            print(f'Allowed return URLs:')
            for url in wi.get('allowedResourceOauth2ReturnUrls', []):
                print(f'  - {url}')
        else:
            print(f'Workload Identity ARN not found in account: {WORKLOAD_IDENTITY_ARN}')
    except Exception as e:
        print(f'WorkloadIdentity API: {e}')
else:
    print('WORKLOAD_IDENTITY_ARN not set — deploy OrgIQ-AgentCoreCapabilities first')

# Example: exchange an external JWT for a workload access token
# (requires a valid JWT from your IdP at runtime)
#
# token = agentcore.get_workload_access_token_for_jwt(
#     identityToken='<your-cognito-or-okta-jwt>',
#     workloadIdentityArn=WORKLOAD_IDENTITY_ARN,
# )
# print(f'Workload token expires: {token["expiresAt"]}')


In [ ]:
config["CODE_EXECUTION_ENABLED"] = True
Path("config.json").write_text(json.dumps(config, indent=2))

print("\u2713 config.json updated")
print("  CODE_EXECUTION_ENABLED = true")

## Summary

- [x] Code interpreter session started and code executed in sandbox
- [x] Strands Agent with `code_interpreter` tool invoked for OrgIQ analysis
- [x] DynamoDB scores analyzed via code execution
- [x] Browser session lifecycle demonstrated (start/navigate/stop)
- [x] Workload Identity ARN verified
- [x] `CODE_EXECUTION_ENABLED = true` saved to config.json

**Next:** [12-ag-ui-streaming.ipynb](12-ag-ui-streaming.ipynb) — AG-UI SSE streaming protocol